# Day 2 · 이론 ③ 결정 트리 — 추가 실습

**공식 실습:** `5-1.ipynb`  
**추가 실습:** `5-1_extra_practice.ipynb`  

실습 A(5-5) → B(6-5) → C(7-5) 순서로 진행합니다. 설명·질문·해석은 강의교안을 참고합니다.

## 0. 공통 데이터 준비

교안의 `train_test_split(test_size=0.2, random_state=42)`와 `ss.fit()` / `ss.transform()`을 사용합니다. 실습 C는 **원본 단위로 학습한** `dt`를 사용합니다.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier

wine = pd.read_csv('https://bit.ly/wine_csv_data')

data = wine[['alcohol', 'sugar', 'pH']]
target = wine['class']

train_input, test_input, train_target, test_target = train_test_split(
    data, target, test_size=0.2, random_state=42
)

ss = StandardScaler()
ss.fit(train_input)
train_scaled = ss.transform(train_input)
test_scaled = ss.transform(test_input)

# 실습 C에서는 표준화하지 않은 모델을 사용합니다.
dt = DecisionTreeClassifier(max_depth=3, random_state=42)
dt.fit(train_input, train_target)

print('훈련 데이터:', train_input.shape)
print('테스트 데이터:', test_input.shape)
print('원본 단위 트리의 훈련 정확도:', round(dt.score(train_input, train_target), 4))

## 실습 A — 지니 불순도와 정보 이득 계산

교안 5-5. 부모·왼쪽·오른쪽 노드의 와인 수를 이용합니다.

### A-1. 부모와 자식 노드의 지니 불순도

In [ ]:
import numpy as np
import pandas as pd

# [레드 와인 수, 화이트 와인 수]
parent_counts = np.array([1258, 3939])
left_counts = np.array([1177, 1745])
right_counts = np.array([81, 2194])

# 지니 불순도 = 1 - (클래스 비율의 제곱 합)
def gini(counts):
    probabilities = counts / counts.sum()
    return 1 - np.sum(probabilities ** 2)

parent_gini = gini(parent_counts)
left_gini = gini(left_counts)
right_gini = gini(right_counts)

print(f'부모 지니 불순도: {parent_gini:.4f}')
print(f'왼쪽 자식 불순도: {left_gini:.4f}')
print(f'오른쪽 자식 불순도: {right_gini:.4f}')

### A-2. 자식 노드의 샘플 수와 정보 이득

In [ ]:
parent_n = parent_counts.sum()
left_n = left_counts.sum()
right_n = right_counts.sum()

print('부모 샘플 수:', parent_n)
print('왼쪽 자식 샘플 수:', left_n)
print('오른쪽 자식 샘플 수:', right_n)

In [ ]:
# 자식 노드의 불순도를 샘플 수에 비례해 가중평균
weighted_gini = (
    (left_n / parent_n) * left_gini
    + (right_n / parent_n) * right_gini
)

# 정보 이득 = 부모 불순도 - 자식 가중평균 불순도
information_gain = parent_gini - weighted_gini

print(f'부모 불순도: {parent_gini:.4f}')
print(f'분할 후 가중평균: {weighted_gini:.4f}')
print(f'정보 이득: {information_gain:.4f}')

### A-3. 학생 응용

A-1의 `left_counts`와 `right_counts`를 각각 `np.array([629, 1970])`, `np.array([629, 1969])`로 **직접 변경**하고 A-1 → A-2 순서로 다시 실행합니다. 이후 기본값으로 되돌려도 됩니다.

## 실습 B — 트리 깊이별 복잡도 비교

교안 6-5. `max_depth`에 따른 실제 깊이·리프 수·훈련 정확도를 확인합니다.

### B-1. 깊이 1·2·3·5·제한 없음 비교

In [ ]:
from sklearn.tree import DecisionTreeClassifier
import pandas as pd

depth_candidates = [1, 2, 3, 5, None]  # None은 깊이 제한 없음

depth_results = []

for depth in depth_candidates:
    # 깊이 설정마다 새 결정 트리를 만들어 학습
    dt_candidate = DecisionTreeClassifier(
        max_depth=depth,
        random_state=42
    )
    dt_candidate.fit(train_scaled, train_target)

    depth_results.append({
        '설정': '제한 없음' if depth is None else depth,
        '실제 깊이': dt_candidate.get_depth(),
        '리프 수': dt_candidate.get_n_leaves(),
        '훈련 정확도': dt_candidate.score(
            train_scaled, train_target
        )
    })

results_b = pd.DataFrame(depth_results)
print(results_b.round(4).to_string(index=False))

### B-2. 실행 결과 비교

결과표에서 `실제 깊이`, `리프 수`, `훈련 정확도`를 비교합니다.

### B-3. 학생 응용

B-1의 `depth_candidates`를 `[1, 2, 3, 4, 5, 7, None]`으로 **직접 변경**한 뒤 B-1을 다시 실행합니다.

## 실습 C — 새로운 와인의 예측 경로와 확률

교안 7-5. **공통 준비에서 원본 단위로 학습한 `dt`**를 사용합니다. `new_wine`은 한 행입니다.

### C-1. 와인 입력값 준비

In [ ]:
new_wine = pd.DataFrame({
    'alcohol': [10.0],
    'sugar': [3.0],  # 이 값만 변경해 다시 실행해 보세요.
    'pH': [3.2]
})

print(new_wine)

### C-2. 예측 클래스와 확률

In [ ]:
prediction = dt.predict(new_wine)
probabilities = dt.predict_proba(new_wine)

print('예측 클래스:', prediction[0])
print('클래스 순서:', dt.classes_)
print('클래스별 확률:', np.round(probabilities[0], 4))

### C-3. 분할 경로 확인

`decision_path()`는 통과한 노드를 알려줍니다.

In [ ]:
node_indicator = dt.decision_path(new_wine)

# 샘플이 하나이므로 통과한 노드 번호가 순서대로 들어 있음
node_ids = node_indicator.indices

feature_names = list(new_wine.columns)

for node_id in node_ids:
    feature_index = dt.tree_.feature[node_id]

    # feature가 음수인 노드는 리프
    if feature_index < 0:
        print(f'리프 노드 {node_id}에 도착')
        continue

    feature_name = feature_names[feature_index]
    threshold = dt.tree_.threshold[node_id]
    input_value = new_wine.iloc[0, feature_index]

    if input_value <= threshold:
        direction = '왼쪽'
    else:
        direction = '오른쪽'

    print(
        f'{feature_name} = {input_value:.3f}, '
        f'기준값 = {threshold:.3f} → {direction}'
    )

### C-4. 리프 번호와 클래스 분포

`apply()`로 리프를 확인하고, 같은 리프의 훈련 샘플 수를 계산합니다.

In [ ]:
leaf_id = dt.apply(new_wine)[0]

print('도착한 리프 노드:', leaf_id)

In [ ]:
# 훈련 샘플마다 도착한 리프 노드 번호
train_leaf_ids = dt.apply(train_input)

# 새 와인과 같은 리프에 도착한 훈련 샘플의 정답만 선택
leaf_targets = train_target[train_leaf_ids == leaf_id]

leaf_counts = leaf_targets.value_counts().sort_index()

print(leaf_counts)
print('리프의 전체 샘플:', len(leaf_targets))

### C-5. 학생 응용

C-1의 `sugar`를 `3.0`에서 `6.0`으로 **직접 변경**하고 C-1 → C-4 순서로 다시 실행합니다. 이후 C-1을 원래 값으로 되돌릴 수 있습니다.